In [1]:
import requests
import pandas as pd
import os
import datetime

## Étape 1 — Extraction / Bronze
Gérer les erreurs lors des appels API : timeout, erreurs HTTP, réponses invalides, etc.

1. Récupérer le dataset des villes marocaines.

In [2]:
ma_url = "https://simplemaps.com/data/ma-cities"
ma_json_url = "https://simplemaps.com/static/data/country-cities/ma/ma.json"

try:
    response = requests.get(url=ma_json_url, timeout=60)
    response.raise_for_status()
    ma_df = pd.DataFrame(data=response.json() or [])
    print(ma_df)
except requests.exceptions.Timeout:
    print("L'API a pris plus d'une minute pour répondre.")
except requests.exceptions.HTTPError as e:
    print(f"Erreur HTTP: {e}")
except requests.exceptions.JSONDecodeError:
    print("Format json invalide.")
# La base des exceptions sourvenues lors d'un request.
except requests.exceptions.RequestException as r:
    print(f"Request Exception: {r}")

                  city      lat       lng  country iso2  \
0           Casablanca  33.5992   -7.6200  Morocco   MA   
1              Tangier  35.7767   -5.8039  Morocco   MA   
2                  Fès  34.0433   -5.0033  Morocco   MA   
3            Marrakech  31.6295   -7.9811  Morocco   MA   
4                 Sale  34.0500   -6.8167  Morocco   MA   
..                 ...      ...       ...      ...  ...   
115        Oulad Yaïch  32.4167   -6.3333  Morocco   MA   
116  Zawyat ech Cheïkh  32.6541   -5.9214  Morocco   MA   
117       Imi-n-Tanout  31.1770   -8.8504  Morocco   MA   
118        Sebt Gzoula  32.1219   -9.0889  Morocco   MA   
119           Tifariti  26.1580  -10.5670  Morocco   MA   

                    admin_name  capital population population_proper  
0            Casablanca-Settat    admin    3950000           3215935  
1    Tanger-Tétouan-Al Hoceïma    admin    1275428           1275428  
2                   Fès-Meknès    admin    1167842           1167842  
3      

2. Utiliser les coordonnées des villes pour interroger l'API Open-Meteo.

In [8]:
meteo_data = []
meteo_url = "https://api.open-meteo.com/v1/forecast"

for lat, lng in ma_df[['lat', 'lng']].to_numpy():
    meteo_params = {
        "latitude": lat,
        "longitude": lng,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "precipitation_probability_max",
            "wind_speed_10m_max",
            "wind_gusts_10m_max"
        ],
        "current": "weather_code",
        "forecast_days": 3
    }
    try:
        response = requests.get(url=meteo_url, params=meteo_params, timeout=300)
        response.raise_for_status()
        meteo_data.append(response.json())
    except requests.exceptions.Timeout:
        print("L'API a pris plus de cinqs minutes pour répondre.")
        break
    except requests.exceptions.HTTPError as e:
        print(f"Erreur HTTP: {e}")
        break
    except requests.exceptions.JSONDecodeError:
        print("Format json invalide.")
        break
    except requests.exceptions.RequestException as r:
        print(f"Request Exception: {r}")
        break

3. Récupérer les prévisions météorologiques quotidiennes des prochains jours.

In [9]:

try:
    meteo_df = pd.DataFrame(data=meteo_data)
except:
    meteo_df = pd.read_csv("bronze/meteo.csv")
meteo_df

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,current_units,current,daily_units,daily
0,33.56250,-7.625000,0.211835,0,GMT,GMT,23.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
1,35.75000,-5.812500,0.336289,0,GMT,GMT,30.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
2,34.06250,-5.000000,0.201464,0,GMT,GMT,389.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
3,31.62500,-8.000000,0.181794,0,GMT,GMT,469.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
4,34.00000,-6.812500,0.179529,0,GMT,GMT,27.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
...,...,...,...,...,...,...,...,...,...,...,...
115,32.43750,-6.312500,1.087189,0,GMT,GMT,503.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
116,32.68750,-5.937500,0.892639,0,GMT,GMT,657.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
117,31.18750,-8.875000,0.195622,0,GMT,GMT,863.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."
118,32.12500,-9.062500,0.265956,0,GMT,GMT,173.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-17T08:30', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-17', '2026-09-18', '2026-09..."


4. Conserver les données brutes dans bronze/.

In [10]:
try:
    os.mkdir(path="./bronze")
    ma_df.to_csv(path_or_buf="./bronze/ma.csv", index=False)
    meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except FileExistsError:
    if "ma.csv" not in os.listdir("./bronze"):
        ma_df.to_csv(path_or_buf="./bronze/ma.csv", index=False)
    if "meteo.csv" not in os.listdir("./bronze"):
        meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except Exception as e:
    print(e)

In [ ]:
# meteo_params_2 = {
#     "latitude": ma_df["lat"].tolist(),
#     "longitude": ma_df["lng"].tolist(),
#     "daily": [
#         "temperature_2m_max",
#         "temperature_2m_min",
#         "precipitation_sum",
#         "precipitation_probability_max",
#         "wind_speed_10m_max",
#         "wind_gusts_10m_max"
#     ],
#     "current": "weather_code",
#     "forecast_days": 3
# }
# response2 = requests.get(
#     url=meteo_url,
#     params=meteo_params_2,
#     timeout=300
# )
# response2.raise_for_status()
# response2.json()

## Étape 2 — Nettoyage / Silver

1. standardiser les types et les dates.

In [5]:
ma_df.dtypes

city                 str
lat                  str
lng                  str
country              str
iso2                 str
admin_name           str
capital              str
population           str
population_proper    str
dtype: object

In [11]:
ma_df["lat"] = ma_df["lat"].astype(float)
ma_df["lng"] = ma_df["lng"].astype(float)
ma_df["population"] = ma_df["population"].astype(float)
ma_df["population_proper"] = ma_df["population_proper"].astype(float)
ma_df.dtypes

city                     str
lat                  float64
lng                  float64
country                  str
iso2                     str
admin_name               str
capital                  str
population           float64
population_proper    float64
dtype: object

In [7]:
meteo_df.dtypes

latitude                 float64
longitude                float64
generationtime_ms        float64
utc_offset_seconds         int64
timezone                     str
timezone_abbreviation        str
elevation                float64
current_units             object
current                   object
daily_units               object
daily                     object
dtype: object

In [12]:
current_units_df = pd.DataFrame(
    data=meteo_df["current_units"].to_list(), 
    index=meteo_df["current_units"].index
)
current_units_df.head(1)

,time,interval,weather_code
0,iso8601,seconds,wmo code


In [11]:
current_units_df.dtypes

time            str
interval        str
weather_code    str
dtype: object

In [13]:
current_df = pd.DataFrame(
    data=meteo_df["current"].to_list(), 
    index=meteo_df["current"].index
)
current_df.head(1)

,time,interval,weather_code
0,2026-09-17T08:30,900,1


In [13]:
current_df.dtypes

time              str
interval        int64
weather_code    int64
dtype: object

In [14]:
current_df["time"] = pd.to_datetime(current_df.time)
current_df.dtypes

time            datetime64[us]
interval                 int64
weather_code             int64
dtype: object

In [15]:
daily_units_df = pd.DataFrame(
    data=meteo_df["daily_units"].to_list(), 
    index=meteo_df["daily_units"].index
)
daily_units_df.head(1)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,iso8601,°C,°C,mm,%,km/h,km/h


In [16]:
daily_units_df.dtypes

time                             str
temperature_2m_max               str
temperature_2m_min               str
precipitation_sum                str
precipitation_probability_max    str
wind_speed_10m_max               str
wind_gusts_10m_max               str
dtype: object

In [16]:
daily_df = pd.DataFrame(
    data=meteo_df["daily"].to_list(), 
    index=meteo_df["daily"].index
)
daily_df.head(1)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,"[2026-09-17, 2026-09-18, 2026-09-19]","[25.4, 25.2, 26.7]","[19.9, 18.3, 19.5]","[0.0, 0.0, 0.0]","[0, 0, 0]","[12.2, 11.9, 11.5]","[35.6, 32.4, 32.0]"


In [18]:
daily_df.columns.to_list()

['time',
 'temperature_2m_max',
 'temperature_2m_min',
 'precipitation_sum',
 'precipitation_probability_max',
 'wind_speed_10m_max',
 'wind_gusts_10m_max']

In [17]:
daily_df_exploded = daily_df.explode(daily_df.columns.to_list())

In [20]:
daily_df_exploded.head(4)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,2026-09-16,25.8,21.2,0.0,0,14.0,35.6
0,2026-09-17,25.7,20.4,0.0,0,12.3,35.3
0,2026-09-18,25.5,19.0,0.0,0,12.7,34.6
1,2026-09-16,25.8,17.6,0.0,0,18.3,42.8


In [21]:
daily_df_exploded.dtypes

time                                str
temperature_2m_max               object
temperature_2m_min               object
precipitation_sum                object
precipitation_probability_max    object
wind_speed_10m_max               object
wind_gusts_10m_max               object
dtype: object

In [18]:
daily_df_exploded.precipitation_probability_max.unique()

array([0, 8, 5, 33, 13, 6, 3, 38, 20, 16, 30, 23, 40, 15, 31, 10, 35, 18,
       14], dtype=object)

In [19]:
# date in iso format 'yyyy-mm-dd'
daily_df_exploded.time = pd.to_datetime(daily_df_exploded.time)
daily_df_exploded.temperature_2m_max = daily_df_exploded.temperature_2m_max.astype(float)
daily_df_exploded.temperature_2m_min = daily_df_exploded.temperature_2m_min.astype(float)
daily_df_exploded.precipitation_sum = daily_df_exploded.precipitation_sum.astype(float)
daily_df_exploded.precipitation_probability_max = (
    daily_df_exploded
    .precipitation_probability_max
    .astype(int)
)
daily_df_exploded.wind_speed_10m_max = daily_df_exploded.wind_speed_10m_max.astype(float)
daily_df_exploded.wind_gusts_10m_max = daily_df_exploded.wind_gusts_10m_max.astype(float)

daily_df_exploded.dtypes

time                             datetime64[us]
temperature_2m_max                      float64
temperature_2m_min                      float64
precipitation_sum                       float64
precipitation_probability_max             int64
wind_speed_10m_max                      float64
wind_gusts_10m_max                      float64
dtype: object

In [20]:
meteo_df.drop(["current_units", "current", "daily_units", "daily"], axis=1, inplace=True)
meteo_df.columns

Index(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds',
       'timezone', 'timezone_abbreviation', 'elevation'],
      dtype='str')

In [21]:
current_df_merged = (
    current_units_df.merge(
        current_df, 
        left_index=True, 
        right_index=True,
        suffixes=("_unit", "")
    )
)
current_df_merged.head(10)

,time_unit,interval_unit,weather_code_unit,time,interval,weather_code
0,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
1,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
2,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,3
3,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
4,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,2
5,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,2
6,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,2
7,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,3
8,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,2
9,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,3


In [22]:
current_df_merged.shape

(120, 6)

In [23]:
daily_df_merged = (
    daily_units_df.merge(
        daily_df_exploded, 
        left_index=True, 
        right_index=True,
        suffixes=("_unit", "")
    )
)
daily_df_merged.head(10)

,time_unit,temperature_2m_max_unit,temperature_2m_min_unit,precipitation_sum_unit,precipitation_probability_max_unit,wind_speed_10m_max_unit,wind_gusts_10m_max_unit,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,25.4,19.9,0.0,0,12.2,35.6
0,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-18,25.2,18.3,0.0,0,11.9,32.4
0,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-19,26.7,19.5,0.0,0,11.5,32.0
1,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,26.9,17.6,0.0,0,10.6,25.6
1,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-18,28.1,19.0,0.0,0,20.3,47.5
1,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-19,27.8,22.5,0.0,0,31.1,74.9
2,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,29.0,18.1,0.0,0,12.0,32.8
2,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-18,31.6,17.9,0.0,0,9.8,26.6
2,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-19,34.3,21.2,0.0,0,18.9,47.9
3,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,30.4,18.4,0.0,0,10.5,31.7


In [30]:
daily_df_merged.shape

(360, 14)

In [24]:
daily_current_df = (
    daily_df_merged.merge(
        current_df_merged, 
        left_index=True, 
        right_index=True,
        suffixes=("_daily", "_current")
    )
)
daily_current_df.head(10)

,time_unit_daily,temperature_2m_max_unit,temperature_2m_min_unit,precipitation_sum_unit,precipitation_probability_max_unit,wind_speed_10m_max_unit,wind_gusts_10m_max_unit,time_daily,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,time_unit_current,interval_unit,weather_code_unit,time_current,interval,weather_code
0,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,25.4,19.9,0.0,0,12.2,35.6,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
0,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-18,25.2,18.3,0.0,0,11.9,32.4,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
0,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-19,26.7,19.5,0.0,0,11.5,32.0,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
1,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,26.9,17.6,0.0,0,10.6,25.6,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
1,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-18,28.1,19.0,0.0,0,20.3,47.5,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
1,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-19,27.8,22.5,0.0,0,31.1,74.9,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
2,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,29.0,18.1,0.0,0,12.0,32.8,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,3
2,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-18,31.6,17.9,0.0,0,9.8,26.6,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,3
2,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-19,34.3,21.2,0.0,0,18.9,47.9,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,3
3,iso8601,°C,°C,mm,%,km/h,km/h,2026-09-17,30.4,18.4,0.0,0,10.5,31.7,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1


In [33]:
daily_current_df.shape

(360, 20)

In [34]:
meteo_df.shape

(120, 7)

In [25]:
meteo_df = (
    meteo_df.merge(
        daily_current_df, 
        left_index=True, 
        right_index=True,
    )
)
# .reset_index(drop=True)
meteo_df.shape

(360, 27)

In [27]:
meteo_df.head()

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,time_unit_daily,temperature_2m_max_unit,temperature_2m_min_unit,...,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,time_unit_current,interval_unit,weather_code_unit,time_current,interval,weather_code
0,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,0.0,0,12.2,35.6,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
0,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,0.0,0,11.9,32.4,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
0,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,0.0,0,11.5,32.0,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
1,35.7500,-5.8125,0.336289,0,GMT,GMT,30.0,iso8601,°C,°C,...,0.0,0,10.6,25.6,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1
1,35.7500,-5.8125,0.336289,0,GMT,GMT,30.0,iso8601,°C,°C,...,0.0,0,20.3,47.5,iso8601,seconds,wmo code,2026-09-17 08:30:00,900,1


In [38]:
meteo_df.dtypes

latitude                                     float64
longitude                                    float64
generationtime_ms                            float64
utc_offset_seconds                             int64
timezone                                         str
timezone_abbreviation                            str
elevation                                    float64
time_unit_daily                                  str
temperature_2m_max_unit                          str
temperature_2m_min_unit                          str
precipitation_sum_unit                           str
precipitation_probability_max_unit               str
wind_speed_10m_max_unit                          str
wind_gusts_10m_max_unit                          str
time_daily                            datetime64[us]
temperature_2m_max                           float64
temperature_2m_min                           float64
precipitation_sum                            float64
precipitation_probability_max                 

2. contrôler la qualité des données.

In [28]:
ma_df[ma_df.duplicated() == True]

,city,lat,lng,country,iso2,admin_name,capital,population,population_proper


In [43]:
ma_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   city               120 non-null    str    
 1   lat                120 non-null    float64
 2   lng                120 non-null    float64
 3   country            120 non-null    str    
 4   iso2               120 non-null    str    
 5   admin_name         120 non-null    str    
 6   capital            120 non-null    str    
 7   population         120 non-null    float64
 8   population_proper  120 non-null    float64
dtypes: float64(4), str(5)
memory usage: 8.6 KB


In [41]:
ma_df.describe()

,lat,lng,population,population_proper
count,120.000000,120.000000,1.200000e+02,1.200000e+02
mean,32.722032,-6.837944,1.747964e+05,1.678812e+05
std,2.148482,2.408281,4.096814e+05,3.529450e+05
min,23.716700,-15.950000,3.000000e+03,3.000000e+03
25%,31.568600,-8.357275,4.019875e+04,4.019875e+04
50%,33.227200,-6.694650,7.558550e+04,7.558550e+04
75%,34.184950,-5.525775,1.279322e+05,1.279322e+05
max,35.841400,-1.911400,3.950000e+06,3.215935e+06


In [29]:
meteo_df[meteo_df.duplicated() == True]

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,time_unit_daily,temperature_2m_max_unit,temperature_2m_min_unit,...,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,time_unit_current,interval_unit,weather_code_unit,time_current,interval,weather_code


In [30]:
meteo_df.isna().sum()

latitude                              0
longitude                             0
generationtime_ms                     0
utc_offset_seconds                    0
timezone                              0
timezone_abbreviation                 0
elevation                             0
time_unit_daily                       0
temperature_2m_max_unit               0
temperature_2m_min_unit               0
precipitation_sum_unit                0
precipitation_probability_max_unit    0
wind_speed_10m_max_unit               0
wind_gusts_10m_max_unit               0
time_daily                            0
temperature_2m_max                    0
temperature_2m_min                    0
precipitation_sum                     0
precipitation_probability_max         0
wind_speed_10m_max                    0
wind_gusts_10m_max                    0
time_unit_current                     0
interval_unit                         0
weather_code_unit                     0
time_current                          0


In [31]:
meteo_df.isnull().sum()

latitude                              0
longitude                             0
generationtime_ms                     0
utc_offset_seconds                    0
timezone                              0
timezone_abbreviation                 0
elevation                             0
time_unit_daily                       0
temperature_2m_max_unit               0
temperature_2m_min_unit               0
precipitation_sum_unit                0
precipitation_probability_max_unit    0
wind_speed_10m_max_unit               0
wind_gusts_10m_max_unit               0
time_daily                            0
temperature_2m_max                    0
temperature_2m_min                    0
precipitation_sum                     0
precipitation_probability_max         0
wind_speed_10m_max                    0
wind_gusts_10m_max                    0
time_unit_current                     0
interval_unit                         0
weather_code_unit                     0
time_current                          0


In [32]:
meteo_df.describe()

,latitude,longitude,generationtime_ms,utc_offset_seconds,elevation,time_daily,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,time_current,interval,weather_code
count,360.000000,360.000000,360.000000,360.0,360.000000,360,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360,360.0,360.000000
mean,32.720886,-6.836718,0.566167,0.0,366.025000,2026-09-18 00:00:00,29.232500,18.342500,0.051944,2.513889,18.197778,36.110833,2026-09-17 08:30:00,900.0,1.825000
min,23.725834,-15.966187,0.133634,0.0,2.000000,2026-09-17 00:00:00,21.100000,13.600000,0.000000,0.000000,8.000000,17.300000,2026-09-17 08:30:00,900.0,0.000000
25%,31.562500,-8.343750,0.204980,0.0,52.750000,2026-09-17 00:00:00,26.675000,16.975000,0.000000,0.000000,14.200000,30.200000,2026-09-17 08:30:00,900.0,1.000000
50%,33.218750,-6.687500,0.338733,0.0,223.500000,2026-09-18 00:00:00,28.800000,18.400000,0.000000,0.000000,16.900000,34.600000,2026-09-17 08:30:00,900.0,2.000000
75%,34.187500,-5.546875,0.767857,0.0,544.250000,2026-09-19 00:00:00,31.500000,19.600000,0.000000,0.000000,20.300000,40.300000,2026-09-17 08:30:00,900.0,3.000000
max,35.812500,-1.937500,3.789306,0.0,1466.000000,2026-09-19 00:00:00,38.400000,26.100000,3.400000,40.000000,41.300000,78.500000,2026-09-17 08:30:00,900.0,3.000000
std,2.139102,2.402353,0.563529,0.0,375.567592,NaN,3.303248,1.978886,0.275856,7.312737,5.618547,9.289556,NaN,0.0,0.998571


3. effectuer la jointure entre les villes et les données météo.

In [33]:
meteo_par_villes_df = meteo_df.merge(ma_df, left_index=True, right_index=True)
meteo_par_villes_df.head()

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,time_unit_daily,temperature_2m_max_unit,temperature_2m_min_unit,...,weather_code,city,lat,lng,country,iso2,admin_name,capital,population,population_proper
0,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,1,Casablanca,33.5992,-7.6200,Morocco,MA,Casablanca-Settat,admin,3950000.0,3215935.0
0,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,1,Casablanca,33.5992,-7.6200,Morocco,MA,Casablanca-Settat,admin,3950000.0,3215935.0
0,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,1,Casablanca,33.5992,-7.6200,Morocco,MA,Casablanca-Settat,admin,3950000.0,3215935.0
1,35.7500,-5.8125,0.336289,0,GMT,GMT,30.0,iso8601,°C,°C,...,1,Tangier,35.7767,-5.8039,Morocco,MA,Tanger-Tétouan-Al Hoceïma,admin,1275428.0,1275428.0
1,35.7500,-5.8125,0.336289,0,GMT,GMT,30.0,iso8601,°C,°C,...,1,Tangier,35.7767,-5.8039,Morocco,MA,Tanger-Tétouan-Al Hoceïma,admin,1275428.0,1275428.0


In [ ]:
meteo_par_villes_df.reset_index(inplace=True, drop=True)
meteo_par_villes_df.head()

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,time_unit_daily,temperature_2m_max_unit,temperature_2m_min_unit,...,weather_code,city,lat,lng,country,iso2,admin_name,capital,population,population_proper
0,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,1,Casablanca,33.5992,-7.6200,Morocco,MA,Casablanca-Settat,admin,3950000.0,3215935.0
1,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,1,Casablanca,33.5992,-7.6200,Morocco,MA,Casablanca-Settat,admin,3950000.0,3215935.0
2,33.5625,-7.6250,0.211835,0,GMT,GMT,23.0,iso8601,°C,°C,...,1,Casablanca,33.5992,-7.6200,Morocco,MA,Casablanca-Settat,admin,3950000.0,3215935.0
3,35.7500,-5.8125,0.336289,0,GMT,GMT,30.0,iso8601,°C,°C,...,1,Tangier,35.7767,-5.8039,Morocco,MA,Tanger-Tétouan-Al Hoceïma,admin,1275428.0,1275428.0
4,35.7500,-5.8125,0.336289,0,GMT,GMT,30.0,iso8601,°C,°C,...,1,Tangier,35.7767,-5.8039,Morocco,MA,Tanger-Tétouan-Al Hoceïma,admin,1275428.0,1275428.0


4. stocker les données nettoyées dans silver/.

In [36]:
try:
    os.mkdir(path="./silver")
    meteo_par_villes_df.to_csv(path_or_buf="./silver/cleaned_meteo_per_town.csv", index=False)
except FileExistsError:
    if "cleaned_meteo_per_town.csv" not in os.listdir("./silver"):
        meteo_par_villes_df.to_csv(
            path_or_buf="./silver/cleaned_meteo_per_town.csv", 
            index=False
        )
except Exception as e:
    print(e)

## Étape 3 — Feature Engineering / Gold

1. catégories de température.

2. catégories de précipitations.

3. catégories de vent.

4. date.

5. autres indicateurs pertinents.

6. Weather Risk Score
<br>
Créer un score de risque météorologique de 0 à 100 permettant d'identifier les conditions potentiellement défavorables.

7. Vous devrez justifier :
* les variables utilisées.
* les seuils.
* la méthode de calcul.

8. Charger les données finales dans PostgreSQL.<br>
Le modèle devra permettre de gérer au minimum :
* les villes et leurs coordonnées.
* les prévisions météorologiques.
* le risk_score.<br>
Vous devrez également prévoir une stratégie pour éviter les doublons lors des nouvelles exécutions du pipeline, car les prévisions peuvent être mises à jour.

9. Bonus:
<br>
conserver l'historique des différentes prévisions.

## Étape 4 — Analyse SQL
Réaliser au minimum 5 requêtes SQL répondant à des questions métier.

1. Quelles villes auront les températures les plus élevées ?

2. Quelles villes auront les plus fortes précipitations ?

3. Quelles villes présentent le risque moyen le plus élevé ?

4. Quelles périodes présentent le risque maximal ?

5. Pour chaque ville, quelle période présente le plus grand risque ?

6. Bonus:
sous-requêtes, fonctions de fenêtrage.

## Étape 5 — Dashboard Streamlit
Créer un dashboard connecté à PostgreSQL permettant de visualiser les prévisions et les risques.
<br>
Le dashboard doit permettre de répondre rapidement à la question :
Où et quand faut-il être particulièrement vigilant dans les prochains jours ?

1. KPI (Key Performance Indicator):
* nombre de villes.
* température maximale.
* précipitations maximales.
* nombre de périodes à risque.
* ville présentant le risque le plus élevé.

2. Filtres:
<br>
Permettre de filtrer notamment par : 
* ville. 
* date. 
* période. 
* niveau de risque.

## Étape 6 — Orchestration & automatisation (Airflow)

1. Définir un
DAG Airflow
qui automatise l'ensemble du pipeline :
* Extraction depuis l'API.
* Nettoyage / transformation.
* Feature engineering & chargement dans PostgreSQL.
* (Rafraîchissement des données pour le dashboard).

2. Planifier une exécution automatique (ex. quotidienne) et gérer les échecs (retries).

3. Conteneuriser le projet avec Docker Compose (Postgres + Airflow + Streamlit).

4. Bonus:
* Ajouter du logging structuré et des alertes en cas d'échec du DAG.
* Historique des prévisions.
* Pipeline incrémental.
* Contrôles de qualité.